
Project : Pentaho Log Intelligence

Layer   : Gold

Notebook: 04_Gold_localhost_accesss

Version : 1.0

Description:
Loads raw Pentaho log files from Unity Catalog Volume

into the Gold Delta table.

Author: Ernesto Felipe Garay Cervantes


#### Recibimiento de parametros

In [0]:
import json

dbutils.widgets.text("archivos_nuevos","")

archivos_nuevos = json.loads(dbutils.widgets.get("archivos_nuevos"))

print("====ARCHIVOS RECIBIDOS===")
for archivo in archivos_nuevos:
    print(archivo) 

#### Configuracion

In [0]:
from pyspark.sql.functions import (
    col,
    lit,
    sha2,
    concat_ws,
    current_timestamp
)

In [0]:
CATALOG = "pentaho_logs"

SILVER_TABLE_LOCALHOST = "pentaho_logs.silver.silver_localhostaccess"

GOLD_TABLE_LOCALHOST= "pentaho_logs.gold.gold_logs_localhostaccess"

#### Lectura  de tabla Bronze 

In [0]:
df_localhost_sl = spark.table(SILVER_TABLE_LOCALHOST).filter(col("file_name").isin(archivos_nuevos))

#display(df_localhost_sl.limit(20))

In [0]:
df_localhost_sl.printSchema()

#### Creación de identificador de Evento Log LocalhostAccess

In [0]:
from pyspark.sql.functions import (col,lit,sha2,concat_ws,current_timestamp)

df_gold_localhost = (df_localhost_sl.withColumn("event_id",sha2(concat_ws("||",col("file_path"),col("hora"),col("aplicacion")),256))
                 .withColumn("source_type",lit("LOCALHOST_ACCESS"))
                  .withColumn("gold_timestamp",current_timestamp())
                 )


In [0]:
#display(
#df_gold_localhost.select(
#          "event_id",
#          "file_name",
#          "aplicacion",
#          "Fecha_evento",
#          "hora",
#          "Peticion_servidor",
 #         "Puerto",
#          "descripcion",
#          "Respuesta_Servidor",
#          "usuario"
# ).limit(20)
#)

In [0]:
df_gold_localhost = df_gold_localhost.select( 
       "event_id",
          "file_name",
          "aplicacion",
          "Fecha_evento",
          "hora",
          "Peticion_servidor",
          "Puerto",
          "descripcion",
          "Respuesta_Servidor",
          "usuario"
 )

In [0]:
#display(df_gold_localhost.limit(20))


#### validación DATAFRAME

In [0]:
# Número de registros
print(f"Total de líneas: {df_gold_localhost.count():,}")

# Estructura
df_gold_localhost.printSchema()

#### Creación Tabla GOLD Pentaho_Log

In [0]:
GOLD_TABLE_LOCALHOST_ACCESS = "pentaho_logs.gold.gold_logs_localhostaccess"
(
    df_gold_localhost.write
        .format("delta")
        .mode("append")
        .saveAsTable(GOLD_TABLE_LOCALHOST_ACCESS)
)

In [0]:
#display(spark.table(GOLD_TABLE_LOCALHOST_ACCESS).limit(20))